<a href="https://colab.research.google.com/github/qaz9391/LINE-BOT-B12090026/blob/main/0709_colab_line_bot_with_gemini_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 20.4 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://blazer-broadly-harpist.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://blazer-broadly-harpist.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology）是一所位於台灣新竹縣新豐鄉的私立科技大學。學校創立於1966年，前身為明新工業專科學校。歷經多次改制，於2002年奉教育部核准升格為明新科技大學，並於2018年更名為「明新學校財團法人明新科技大學」。

學校校名「明新」取自《大學》「在明明德，在新民，在止於至善」之精義，旨在闡揚人類與生俱來的德性與情操，並以培育「堅毅、求新、創造」精神的專業人才為辦學目標。

明新科技大學現設有半導體學院、工程學院、管理學院、民生學院、人文與設計學院、共同教育學院等六個學院，涵蓋多個學系、學位學程及碩士班，並於2022年獲准設立半導體科技博士學位學程。

該校地理位置鄰近新竹科學園區與新竹工業區，享有豐富的產業資源，因此致力於以「產業大學」為定位，積極推動產學合作。學校特別鎖定半導體、AI、元宇宙、風電綠能等前瞻產業，發展「MUST」四大育才特色，包含多元學習（Multidisciplinary Learning）、全球視野（Universal Perspective）、永續經營（Sustainable Operations）與技術創新（Technological Innovation），旨在培育具備跨域整合、務實創新與全人學習能力的專業人才，以符合產業需求。明新科技大學的畢業生在業界表現優異，曾獲企業好評與肯定。


In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

目前明新科技大學的校長是**呂明峯**。

他於2025年1月16日舉行了第11任校長布達暨交接典禮，並於2025年2月1日正式上任。呂明峯校長具備豐富的業界經驗和學術背景，並曾打造全台首座半導體封裝測試類產線，推動成立半導體學院。


In [ ]:
from flask import Flask, request, abort
import logging
import os
import time
from google.genai import types

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    MessagingApiBlob,
    ReplyMessageRequest,
    TextMessage
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
    FileMessageContent
)

app = Flask(__name__)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
app.logger.setLevel(logging.INFO)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)

# 儲存檔案的目錄
UPLOAD_DIR = "/content/uploaded_files"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# 儲存每個使用者的對話 session 和上傳的檔案
user_sessions = {}  # {user_id: {"chat": chat_object, "uploaded_file": gemini_file}}

def get_user_session(user_id):
    """取得或建立使用者的對話 session"""
    if user_id not in user_sessions:
        # 建立新的對話 session
        new_chat = client.chats.create(
            model="gemini-2.5-flash",
            config=GenerateContentConfig(
                system_instruction="你是一個中文的AI助手，請用繁體中文回答。如果使用者有提供參考文件，請根據文件內容回答問題。",
                tools=[google_search_tool],
                response_modalities=["TEXT"],
            )
        )
        user_sessions[user_id] = {
            "chat": new_chat,
            "uploaded_file": None
        }
    return user_sessions[user_id]

def download_line_file(message_id, file_name):
    """從 LINE 下載使用者上傳的檔案"""
    with ApiClient(configuration) as api_client:
        line_bot_blob_api = MessagingApiBlob(api_client)
        file_content = line_bot_blob_api.get_message_content(message_id)

        file_path = os.path.join(UPLOAD_DIR, file_name)

        with open(file_path, 'wb') as f:
            f.write(file_content)

        return file_path

def upload_file_to_gemini(file_path):
    """上傳檔案到 Gemini Files API"""
    uploaded_file = client.files.upload(
        file=file_path,
        config={'display_name': os.path.basename(file_path)}
    )

    # 等待檔案處理完成
    while uploaded_file.state.name == "PROCESSING":
        print("檔案處理中...")
        time.sleep(1)
        uploaded_file = client.files.get(name=uploaded_file.name)

    if uploaded_file.state.name == "FAILED":
        raise Exception("檔案上傳處理失敗")

    return uploaded_file

def query_with_rag(user_id, question):
    """使用 RAG 模式回答問題"""
    session = get_user_session(user_id)
    uploaded_file = session["uploaded_file"]

    if uploaded_file:
        # 有上傳檔案，使用 RAG 模式
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_uri(
                            file_uri=uploaded_file.uri,
                            mime_type=uploaded_file.mime_type
                        ),
                        types.Part.from_text(text=f"請根據上述提供的檔案內容，用繁體中文回答這個問題：{question}")
                    ]
                )
            ]
        )
        return response.text
    else:
        # 沒有上傳檔案，使用一般多輪對話
        response = session["chat"].send_message(message=question)
        return response.text

@app.route("/", methods=['POST'])
def callback():
    signature = request.headers['X-Line-Signature']
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature.")
        abort(400)

    return 'OK'

@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    """處理文字訊息"""
    text = event.message.text
    user_id = event.source.user_id

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        if text.startswith('AI '):
            prompt = text[3:]
            try:
                # 使用 RAG 或一般對話
                reply_text = query_with_rag(user_id, prompt)

                # 檢查是否有上傳檔案，加上提示
                session = get_user_session(user_id)
                if session["uploaded_file"]:
                    reply_text = f"📄 [RAG 模式]\n\n{reply_text}"

                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=reply_text)]
                    )
                )
            except Exception as e:
                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=f"❌ 發生錯誤：{str(e)}")]
                    )
                )
        elif text == "清除文件":
            # 清除使用者上傳的檔案
            session = get_user_session(user_id)
            session["uploaded_file"] = None
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="✅ 已清除上傳的文件，恢復一般對話模式。")]
                )
            )
        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="請輸入「AI 問題」來開始對話\n或上傳 TXT/PDF 檔案啟用 RAG 模式")]
                )
            )

@handler.add(MessageEvent, message=FileMessageContent)
def handle_file_message(event):
    """處理使用者上傳的檔案"""
    user_id = event.source.user_id
    file_name = event.message.file_name

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # 檢查檔案類型
        if not (file_name.endswith('.txt') or file_name.endswith('.pdf')):
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="⚠️ 目前只支援 TXT 或 PDF 檔案")]
                )
            )
            return

        try:
            # 下載檔案
            file_path = download_line_file(event.message.id, file_name)
            print(f"檔案已下載：{file_path}")

            # 上傳到 Gemini
            uploaded_file = upload_file_to_gemini(file_path)
            print(f"檔案已上傳到 Gemini：{uploaded_file.uri}")

            # 儲存到使用者 session
            session = get_user_session(user_id)
            session["uploaded_file"] = uploaded_file

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"✅ 檔案「{file_name}」上傳成功！\n\n現在您可以輸入「AI 問題」來詢問關於這份文件的問題。\n\n輸入「清除文件」可恢復一般對話模式。")]
                )
            )
        except Exception as e:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"❌ 檔案處理失敗：{str(e)}")]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:__main__:Request body: {"destination":"Ue9bc54ad0cac458e5ed676a3e8751c97","events":[{"type":"message","message":{"type":"text","id":"616777342916493824","quoteToken":"PXrPLbVG15xBbUeiat4sgb1cdvJITbIvwYgqcpZejHlV6HjlccAKYvGeygCcpl4XxSF9S8DUg3zSSrhxXQJyGCbAMmdTcuiUI3rAHcvL40fNYoV0EAh2AOZplbQ9F0v-IjN7NrJ4DARre1Wlaz5pCg","markAsReadToken":"BfAvUzeKRopU9xzjVeFHcW5r5TkOkP_h5Jf2Uq3vrL4oY-50lnCsSNG-XVZDOjxYqeziLJHhlPq53P2dkeYjz54w3eeLoIi39ZGjOATru-D_xkluN32TbdzAzOrujYh1RFbkr21T326xaY_9wc6du3QScj8trEo6O_rkmi8eUVydAZoIYj79Fn8tBjEj9G_JXE5QGnTSJxlJe7KARKNDpQ","text":"AI 校長愛吃甚麼?"},"webhookEventId":"01KT5T870Y76KQW2AJBPCZJ9RW","deliveryContext":{"isRedelivery":false},"timestamp":1780459248356,"source":{"type":"user","userId":"U4a7effcc683b4b1daf00f53459845948"},"replyToken":"7c035a6f2557465ea51

BODY:  {"destination":"Ue9bc54ad0cac458e5ed676a3e8751c97","events":[{"type":"message","message":{"type":"text","id":"616777342916493824","quoteToken":"PXrPLbVG15xBbUeiat4sgb1cdvJITbIvwYgqcpZejHlV6HjlccAKYvGeygCcpl4XxSF9S8DUg3zSSrhxXQJyGCbAMmdTcuiUI3rAHcvL40fNYoV0EAh2AOZplbQ9F0v-IjN7NrJ4DARre1Wlaz5pCg","markAsReadToken":"BfAvUzeKRopU9xzjVeFHcW5r5TkOkP_h5Jf2Uq3vrL4oY-50lnCsSNG-XVZDOjxYqeziLJHhlPq53P2dkeYjz54w3eeLoIi39ZGjOATru-D_xkluN32TbdzAzOrujYh1RFbkr21T326xaY_9wc6du3QScj8trEo6O_rkmi8eUVydAZoIYj79Fn8tBjEj9G_JXE5QGnTSJxlJe7KARKNDpQ","text":"AI 校長愛吃甚麼?"},"webhookEventId":"01KT5T870Y76KQW2AJBPCZJ9RW","deliveryContext":{"isRedelivery":false},"timestamp":1780459248356,"source":{"type":"user","userId":"U4a7effcc683b4b1daf00f53459845948"},"replyToken":"7c035a6f2557465ea515417c8645f354","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [03/Jun/2026 04:00:50] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"Ue9bc54ad0cac458e5ed676a3e8751c97","events":[{"type":"message","message":{"type":"file","id":"616777370363232749","markAsReadToken":"e-n0FNRWufizYV4M_SXRqtRBrxoxi9vKEw76oEWX0sQWqQBHC_W9yCA1sunScJZtIK4gQscwMA1Y0K3PqoZa4y_vFS-G1UTHa8NjUj3g853tBX1-DqRvcFr3Fbo2xC97Sk7mpRf4Z-IF00wStFTVvwB7FBb9v3WyaOhetGzqFA5XgMFlByAYK3Ef8vZYEb-ZWMSnOzu1EF9qC--Ot-9j-Q","fileName":"RAG.txt","fileSize":21,"contentProvider":{"type":"line"}},"webhookEventId":"01KT5T8Q7A1W60DBBZS7R6DN3Q","deliveryContext":{"isRedelivery":false},"timestamp":1780459264737,"source":{"type":"user","userId":"U4a7effcc683b4b1daf00f53459845948"},"replyToken":"d5a128e5eaf94c8f91676d376c44e203","mode":"active"}]}


BODY:  {"destination":"Ue9bc54ad0cac458e5ed676a3e8751c97","events":[{"type":"message","message":{"type":"file","id":"616777370363232749","markAsReadToken":"e-n0FNRWufizYV4M_SXRqtRBrxoxi9vKEw76oEWX0sQWqQBHC_W9yCA1sunScJZtIK4gQscwMA1Y0K3PqoZa4y_vFS-G1UTHa8NjUj3g853tBX1-DqRvcFr3Fbo2xC97Sk7mpRf4Z-IF00wStFTVvwB7FBb9v3WyaOhetGzqFA5XgMFlByAYK3Ef8vZYEb-ZWMSnOzu1EF9qC--Ot-9j-Q","fileName":"RAG.txt","fileSize":21,"contentProvider":{"type":"line"}},"webhookEventId":"01KT5T8Q7A1W60DBBZS7R6DN3Q","deliveryContext":{"isRedelivery":false},"timestamp":1780459264737,"source":{"type":"user","userId":"U4a7effcc683b4b1daf00f53459845948"},"replyToken":"d5a128e5eaf94c8f91676d376c44e203","mode":"active"}]}
檔案已下載：/content/uploaded_files/RAG.txt
檔案已上傳到 Gemini：https://generativelanguage.googleapis.com/v1beta/files/a4p0s908igj4


INFO:werkzeug:127.0.0.1 - - [03/Jun/2026 04:01:07] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"Ue9bc54ad0cac458e5ed676a3e8751c97","events":[{"type":"message","message":{"type":"text","id":"616777386452582916","quoteToken":"_j4n465ZbOUbHYioQbGAJqZKvC1DjrtuNPS9rgRuZqlUiHGSotVlgeCg-mW_fdiQ3Jl6gMOqpVYxVxjqsQ6wlOUULQo4bJBspUXdeBISs_X3H5gDTlm1DaSHSHxA-dHFsRUiVW7H5cGeniH57U0o_A","markAsReadToken":"af00ZhRdhFBFXZRulIoD0ut6jylVh01IBzGbcYAPITosP7VqtMWrBfIoRvlKGdWvnRwBs7dE9vF2_n_VqrJwn9UM3lGCtl-Bimvtk-SEgsudadmPoJTQ-ZJPfovfPpkV0JODm9An61EiXFFssdd3AA_WVcLYApyfoJAJz2F9vytbB1UbRMeMZG47bnGBSr-jzAidMSZc_DdDDx-YUw9g-g","text":"AI 校長愛吃甚麼?"},"webhookEventId":"01KT5T905T1YKV1KTT123DVMGG","deliveryContext":{"isRedelivery":false},"timestamp":1780459274343,"source":{"type":"user","userId":"U4a7effcc683b4b1daf00f53459845948"},"replyToken":"7e420ce43b9c4e1d88d6b96877e05693","mode":"active"}]}


BODY:  {"destination":"Ue9bc54ad0cac458e5ed676a3e8751c97","events":[{"type":"message","message":{"type":"text","id":"616777386452582916","quoteToken":"_j4n465ZbOUbHYioQbGAJqZKvC1DjrtuNPS9rgRuZqlUiHGSotVlgeCg-mW_fdiQ3Jl6gMOqpVYxVxjqsQ6wlOUULQo4bJBspUXdeBISs_X3H5gDTlm1DaSHSHxA-dHFsRUiVW7H5cGeniH57U0o_A","markAsReadToken":"af00ZhRdhFBFXZRulIoD0ut6jylVh01IBzGbcYAPITosP7VqtMWrBfIoRvlKGdWvnRwBs7dE9vF2_n_VqrJwn9UM3lGCtl-Bimvtk-SEgsudadmPoJTQ-ZJPfovfPpkV0JODm9An61EiXFFssdd3AA_WVcLYApyfoJAJz2F9vytbB1UbRMeMZG47bnGBSr-jzAidMSZc_DdDDx-YUw9g-g","text":"AI 校長愛吃甚麼?"},"webhookEventId":"01KT5T905T1YKV1KTT123DVMGG","deliveryContext":{"isRedelivery":false},"timestamp":1780459274343,"source":{"type":"user","userId":"U4a7effcc683b4b1daf00f53459845948"},"replyToken":"7e420ce43b9c4e1d88d6b96877e05693","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [03/Jun/2026 04:01:17] "POST / HTTP/1.1" 200 -
